In [1]:
!pip install google-genai chromadb pypdf tqdm

   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
   -------------------- ------------------- 12.3/23.5 MB 60.6 MB/s eta 0:00:01
   ---------------------------------------- 23.5/23.5 MB 58.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/4.9 MB ? eta -:--:--
   ---------------------------------------- 4.9/4.9 MB 56.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/4.6 MB ? eta -:--:--
   ---------------------------------------- 4.6/4.6 MB 52.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/13.4 MB ? eta -:--:--
   ------------------------------------- -- 12.6/13.4 MB 61.8 MB/s eta 0:00:01
   ---------------------------------------- 13.4/13.4 MB 48.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 2.8/2.8 MB 25.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/719.8 kB ? eta -:--:--
   --------------------------------

  You can safely remove it manually.
  You can safely remove it manually.


In [3]:
import os 
from pypdf import PdfReader

#Configurar la lectura del PDF

RUTA_PDF="Apuntes silvia.pdf"

def procesar_y_fragmentar_pdf(ruta_archivo, chunk_size=1000, chunk_overlap=200):
    """Lee un PDF, lo fragmnenta en trozos (chunks) guardando el bloque de texto
    y numero como metadato"""
    if not os.path.exist(ruta_archivo):
        raise FileNotFoundError(f"No se encntró el archibvo en: {ruta_archivo}")
    reader=PdfReader(ruta_archivo)
    chunks_del_proyecto=[]
    print(f"Leyendo {len(reader.pages)} páginas del PDF")

#Ahora recorremos el PDF página por página

for num_pagina, pagina in enumerate(reader.pages, start=1):
    texto_pagina=pagina.extract_text()
    if not texto_pagina:
        continue
    #Iniciamos el Chunking con solapamiento sobre el texto
    inicio=0
    while inicio < len (texto_pagina):
        fin=inicio+chunk_size
        fragmento_texto=texto_pagina[inicio:fin]

        #Guardamos la estructura limpia para nuestro vector store
        chunks_del_proyecto_append({
        'texto':fragmento_texto.strip(),
        'metadato':{'páginas':num_paginas, 'fuente':os.path.basename(ruta_archivo)})

        #Desplazamos el puntero teniendo en cuenta el solapamiento
        inicio +=(chunk_size - chunck_overlap)
    print(f"Fragmentación completada, se han generado {len(chucks_del_proyecto}")
    return chucks_del_proyecto

#Ejecutamos la función
try:
    chucks=procesar_y_fragmentar_pdf(RUTA_PDF)

    #Sacamos una muestra del primer chunk para ver si funciona
    print(f"\n MUESTRA del chunk 1:")
    print(f"Página de origen: {chunks[0]['metadato']['páginas']}")
except Exception as e:
    print(e)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (4293875148.py, line 32)

In [5]:
import os 
from pypdf import PdfReader

# Configurar la lectura del PDF
RUTA_PDF = "Apuntes Marketing/Apuntes silvia.pdf"

def procesar_y_fragmentar_pdf(ruta_archivo, chunk_size=1000, chunk_overlap=200):
    """
    Lee un PDF, lo fragmenta en trozos (chunks) guardando el bloque de texto
    y número como metadato.
    """
    if not os.path.exists(ruta_archivo): 
        raise FileNotFoundError(f"No se encontró el archivo en: {ruta_archivo}")
        
    reader = PdfReader(ruta_archivo)
    chunks_del_proyecto = []
    print(f"📖 Leyendo {len(reader.pages)} páginas del PDF...")

    # Ahora recorremos el PDF página por página 
    for num_pagina, pagina in enumerate(reader.pages, start=1):
        texto_pagina = pagina.extract_text()
        if not texto_pagina:
            continue
            
        # Iniciamos el Chunking con solapamiento sobre el texto
        inicio = 0
        while inicio < len(texto_pagina):
            fin = inicio + chunk_size
            fragmento_texto = texto_pagina[inicio:fin]

            # Guardamos la estructura limpia para nuestro vector store (Corregido: .append y paréntesis de cierre)
            chunks_del_proyecto.append({
                'texto': fragmento_texto.strip(),
                'metadato': {'página': num_pagina, 'fuente': os.path.basename(ruta_archivo)}
            })

            # Desplazamos el puntero teniendo en cuenta el solapamiento 
            inicio += (chunk_size - chunk_overlap)
            
    print(f"🧱 Fragmentación completada, se han generado {len(chunks_del_proyecto)} chunks.")
    return chunks_del_proyecto

# Ejecutamos la función
try:
    chunks = procesar_y_fragmentar_pdf(RUTA_PDF) 

    # Sacamos una muestra del primer chunk para ver si funciona
    if chunks:
        print(f"\n🔬 MUESTRA del chunk 1:")
        print(f"📄 Página de origen: {chunks[0]['metadato']['página']}")
        print(f"📝 Texto:\n{chunks[0]['texto'][:200]}...")
    else:
        print("⚠️ El PDF se leyó pero no se generaron fragmentos (¿está vacío o es solo imagen?).")
        
except Exception as e:
    print(f"❌ Ocurrió un error: {e}")

📖 Leyendo 30 páginas del PDF...
🧱 Fragmentación completada, se han generado 87 chunks.

🔬 MUESTRA del chunk 1:
📄 Página de origen: 1
📝 Texto:
Tema 1: Concepto y Naturaleza del Marketing Financiero. 
El Marketing es un término habitualmente usado en el mundo financiero y empresarial sin un 
significado uniforme. Se puede entender de varias m...


In [ ]:
import os
import chroma db
from google import genai

#Iniciamos cliente

client=genai.Cliente()

RUTA_DB="db_marketing"

print(f"Iniciando conexión con Chroma")

#Se crea un cliente en chroma para guardar los datos

chroma_client=chromadb.PersistenClient(path=RUTA_DB)

#Creamos una colección para los apuntes
coleccion=chroma_client.get_or_create_collection(name='apuntes marketing')

print(f"Generando Embedding y indexando base de datos")

#Ahora vamos a recorrer los chunks que creamos en el paso anterior

for i, chunk in enumerate(chunks):
    try:
        #Enviamos texto a Gemini para que convierta en vector
        respuesta_embedding=cliente_gemini.models.compute_embeddings(
            model='text-embedding-004',
            contents=chunk['texto']
        )

        #Extraemos los valores númericos del Embedding
        vector= respuesta_embedding.embeddings[0].values

        #Guardamos en Chroma todo (vector+texto original+Metadado)
        coleccion.add(
            embeddings=[vector],
            documents=[chunk['texto']],
            metadatas=[chunk['metadato']]
            ids=[f"id_{i}]"]
        )

    except Exception as e:
        print(f"Error procesando el Chunk {i}: {e}")
        break

    print(f"Se ha indexado {coleccion.count()} fragmentos a tu base de datos "
    

In [11]:
import os
import chromadb

# 1. Configurar la ruta hacia tu carpeta db_marketing
RUTA_DB = "db_marketing"

print("Conectando con ChromaDB")
chroma_client = chromadb.PersistentClient(path=RUTA_DB)

# 2. Creamos o cargamos la colección
#Chroma usa automáticamente su modelo interno de embeddings locales
coleccion = chroma_client.get_or_create_collection(name='apuntes_marketing')

print("Indexando los fragmentos en la base de datos local")

# Extraemos los textos, metadatos e IDs de nuestra lista de chunks
lista_textos = [chunk['texto'] for chunk in chunks]
lista_metadatos = [chunk['metadato'] for chunk in chunks]
lista_ids = [f"id_{i}" for i in range(len(chunks))]

try:
    #Chroma calcula los embeddings e indexa todo en UNA sola línea de código
    coleccion.add(
        documents=lista_textos,
        metadatas=lista_metadatos,
        ids=lista_ids
    )
    print(f"\n Se han indexado {coleccion.count()} fragmentos en tu base de datos vectorial")
    
except Exception as e:
    print(f"Ocurrió un error con Chroma: {e}")

Conectando con ChromaDB
Indexando los fragmentos en la base de datos local


C:\Users\diazm\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:13<00:00, 6.21MiB/s]



 Se han indexado 87 fragmentos en tu base de datos vectorial


In [14]:
from google import genai

# 1. Iniciamos el cliente
client = genai.Client()

def consultar_chatbot_rag(pregunta_usuario):
    """
    Flujo RAG completo: Recupera de Chroma, arma el contexto y responde con Gemini.
    """
    # Fase 1: Retrieval 
    # Buscamos en Chroma los 3 fragmentos (n_results=10) matemáticamente más cercanos a la pregunta
    resultados = coleccion.query(
        query_texts=[pregunta_usuario],
        n_results=10
    )
    
    # Extraemos los textos recuperados y sus páginas
    fragmentos_encontrados = resultados['documents'][0]
    metadatos_encontrados = resultados['metadatas'][0]
    
    # PASO 2: Contexto 
    # Unimos los fragmentos encontrados en un solo bloque de texto limpio
    contexto_apuntes = ""
    for idx, texto in enumerate(fragmentos_encontrados):
        pagina = metadatos_encontrados[idx]['página']
        contexto_apuntes += f"\n--- [FRAGMENTO DE APUNTES - PÁGINA {pagina}] ---\n{texto}\n"
        
    # Diseñamos las instrucciones estrictas para Gemini
    system_instruction = (
        "Eres un asistente experto en Marketing Financiero. Tu objetivo es responder "
        "a la pregunta del usuario utilizando ÚNICAMENTE la información provista en el CONTEXTO. "
        "Si la respuesta no está clara o no viene en el contexto, di amablemente: "
        "'Lo siento, no encuentro esa información específica en tus apuntes de marketing.' "
        "No inventes datos ni uses conocimiento externo que contradiga o complemente de más tus apuntes."
    )
    
    prompt_completo = f"""
    CONTEXTO DE MIS APUNTES:
    {contexto_apuntes}
    
    PREGUNTA DEL USUARIO:
    {pregunta_usuario}
    """
    
    # PASO 3: Respuesta del Modelo
    try:
        
        respuesta = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt_completo,
            config={'system_instruction': system_instruction}
        )
        
        # Imprimimos la respuesta en pantalla
        print("\n CHATBOT RAG:")
        print(respuesta.text)
        print("\n FUENTES UTILIZADAS:")
        paginas_utilizadas = set([m['página'] for m in metadatos_encontrados])
        print(f"    Información extraída de las páginas: {sorted(list(paginas_utilizadas))}")
        
    except Exception as e:
        print(f"❌ Error al generar la respuesta con Gemini: {e}")

# Hazle una pregunta real sobre el Tema 1 (vimos que hablaba sobre Concepto y Naturaleza del Marketing Financiero)
pregunta = "¿Qué se entiende por Marketing Financiero según el Tema 1?"
print(f"Preguntando al sistema: '{pregunta}'")
consultar_chatbot_rag(pregunta)

Preguntando al sistema: '¿Qué se entiende por Marketing Financiero según el Tema 1?'

 CHATBOT RAG:
Según el Tema 1 de tus apuntes, la definición más correcta de Marketing para el profesor es:

El Marketing es la actividad y/o conjunto de procesos llevados a cabo por una organización y que van dirigidos a identificar y satisfacer las necesidades de los consumidores, usuarios y de la sociedad, y que tiene por objetivo la creación de valor para todos los agentes.

 FUENTES UTILIZADAS:
    Información extraída de las páginas: [1, 2, 7, 8, 10, 11, 21, 23]


In [15]:
import sys


print("ℹ️  Escribe tu pregunta y presiona Enter. (Para cerrar el chat, escribe: salir)\n")

# Iniciamos el bucle interactivo
while True:
    # 1. Capturamos la pregunta del usuario en tiempo real
    pregunta_usuario = input("👤 Tú: ")
    
    # 2. Condición de salida: si escribe "salir", rompemos el bucle
    if pregunta_usuario.strip().lower() == "salir":
        print("\n Chatbot: ¡Hasta luego!")
        break
        
    # 3. Controlamos que no envíe una pregunta vacía por error
    if not pregunta_usuario.strip():
        print("Por favor, escribe una pregunta válida.")
        continue
        
    # 4. Ejecutamos la función RAG que programamos antes
    print("Buscando en tus apuntes y procesando con Gemini...")
    consultar_chatbot_rag(pregunta_usuario)
    print("-" * 60) # Línea divisoria para separar las respuestas

ℹ️  Escribe tu pregunta y presiona Enter. (Para cerrar el chat, escribe: salir)



👤 Tú:  Que es el marketing operativo


Buscando en tus apuntes y procesando con Gemini...

 CHATBOT RAG:
El marketing operativo (cp) se ocupa de establecer las acciones concretas que deben realizarse en el corto o medio plazo para conseguir los objetivos establecidos (LAS 4 Ps).

 FUENTES UTILIZADAS:
    Información extraída de las páginas: [1, 8, 18, 19, 22, 23, 27, 28, 29]
------------------------------------------------------------


👤 Tú:  salir



 Chatbot: ¡Hasta luego!
